In [12]:
import pdfplumber
import pandas as pd

# 1. Open the PDF
with pdfplumber.open("annexure5.pdf") as pdf:
    all_data = []
    
    # 2. Iterate through pages
    for page in pdf.pages:
        # extract_table automatically finds the grid lines
        table = page.extract_table()
        if table:
            all_data.extend(table)

# 3. Clean and Save
# The first row is usually your header
df = pd.DataFrame(all_data[1:], columns=all_data[0])

# Remove the '\n' characters that PDFs love to insert
df = df.replace('\n', ' ', regex=True)

df.to_csv("Bengaluru_Tree_Palette.csv", index=False)
print("Done! Check your folder for Bengaluru_Tree_Palette.csv")

Done! Check your folder for Bengaluru_Tree_Palette.csv


In [13]:
# 1. Open the PDF
with pdfplumber.open("CES_TVR_ETR75_TREES_24may2014.pdf") as pdf:
    all_data = []
    
    # 2. Iterate through pages 33 to 38 (Indices 32 to 38)
    # pdf.pages[32:38] will take pages 33, 34, 35, 36, 37, and 38
    target_pages = pdf.pages[32:38]
    
    for i, page in enumerate(target_pages):
        table = page.extract_table()
        if table:
            # If it's the first target page, keep the header
            if i == 0:
                all_data.extend(table)
            else:
                # For subsequent pages, skip the header row (table[1:])
                all_data.extend(table[1:])

# 3. Clean and Save
# Create DataFrame using the first row of all_data as the header
df = pd.DataFrame(all_data[1:], columns=all_data[0])

# Clean the '\n' characters and extra spaces
df = df.replace(r'\n', ' ', regex=True).replace(r'\s+', ' ', regex=True)

# Drop any completely empty rows
df = df.dropna(how='all')

df.to_csv("Ward_wise_tree_details.csv", index=False)
print(f"Done! Extracted {len(df)} wards into Ward_wise_tree_details.csv")

Done! Extracted 195 wards into Ward_wise_tree_details.csv


In [20]:
import re

# 1. Open the extracted PDF
pdf_path = "CES_TVR_ETR75_TREES_24may2014 Extract[38-48].pdf"

all_tree_data = []

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        if not text:
            continue
            
        # Each tree entry starts with 'Common name:', so we split the text into blocks
        # using a lookahead to keep the 'Common name:' marker
        blocks = re.split(r'(?=Common name:)', text)
        
        for block in blocks:
            if "Common name:" in block:
                # Regex patterns for your specific columns
                # [\s\S]*? allows matching across multiple lines (for Description)
                record = {
                    "Common_Name": re.search(r'Common name:\s*(.*)', block),
                    "Family": re.search(r'Family:\s*(.*)', block),
                    "Description": re.search(r'Description:\s*([\s\S]*?)(?=Flowering:|$)', block),
                    "Flowering": re.search(r'Flowering:\s*(.*)', block),
                    "Native": re.search(r'Native:\s*(.*)', block),
                    "Location": re.search(r'Location:\s*(.*)', block)
                }
                
                # Clean the results: extract string and replace mid-text newlines with spaces
                clean_record = {}
                for key, match in record.items():
                    if match:
                        val = match.group(1).replace('\n', ' ').strip()
                        clean_record[key] = val
                    else:
                        clean_record[key] = "N/A"
                
                all_tree_data.append(clean_record)

# 2. Create and Clean DataFrame
df_annex3 = pd.DataFrame(all_tree_data)

# Remove duplicates (if any species span across page breaks)
df_annex3 = df_annex3.drop_duplicates(subset=['Common_Name'])

# 3. Save to CSV
df_annex3.to_csv("Annexure3_Prominent_Trees.csv", index=False)

print(f"Success! Extracted {len(df_annex3)} prominent trees into Annexure3_Prominent_Trees.csv")
print(df_annex3[['Common_Name', 'Family', 'Native']].head())

Success! Extracted 40 prominent trees into Annexure3_Prominent_Trees.csv
            Common_Name       Family                   Native
0     Australian wattle     Fabaceae                Australia
1        Butterfly tree     Fabaceae    India, Burma, Vietnam
2  Red silk-cotton tree  Bombacaceae                    India
3    Popcorn bush cedar     Fabaceae  Tropical Southeast Asia
4          Coconut palm    Arecaceae             Indo-Pacific
